In [ ]:
from google.colab import drive
import os
import shutil
import sys
from pathlib import Path

# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=True)

# ── Step 2: Find and unzip the dataset zip ───────────────────────────────────
# Upload  heritagelens-data.zip  to Google Drive → MyDrive  before running this.
zip_name   = "heritagelens-data.zip"
candidates = [
    Path("/content/drive/MyDrive") / zip_name,
    Path("/content") / zip_name,
]

target_zip = None
for p in candidates:
    if p.exists():
        target_zip = p
        break

if target_zip:
    print(f"Found: {target_zip}")
    print("Unzipping into /content/ ...")
    try:
        shutil.unpack_archive(str(target_zip), "/content/", "zip")
        print("Unzip complete.")
    except OSError as e:
        if getattr(e, "errno", None) == 107:
            print("Transport endpoint error — restart the runtime and try again.")
        raise
else:
    print("WARNING: heritagelens-data.zip not found.")
    print("Upload it to Google Drive → MyDrive and rerun this cell.")

# ── Step 3: Set ROOT ──────────────────────────────────────────────────────────
# The zip extracts as  data/  at /content/ → we use /content as ROOT.
ROOT = Path("/content")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

# Verify data is present
data_dir = ROOT / "data"
if data_dir.exists():
    print(f"\nROOT = {ROOT}")
    print("data/ contents:", sorted(x.name for x in data_dir.iterdir()))
    print("\n✓ Ready — run the next cells in order.")
else:
    print("\nWARNING: data/ folder not found after unzip. Check the zip structure.")

In [ ]:
# ── Cell 1: Install libraries & global setup ─────────────────────────────────
!pip install -q transformers datasets torch torchvision tqdm nltk

import sys, os, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm

try:
    from google.colab import userdata
    from huggingface_hub import login
    try:
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            login(hf_token)
            print("Logged in to Hugging Face Hub.")
        else:
            print("No HF_TOKEN — unauthenticated access (fine for gpt2).")
    except Exception:
        print("Using unauthenticated access to Hugging Face Hub.")
except Exception:
    pass

if "ROOT" not in dir():
    ROOT = Path("/content")
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    os.chdir(ROOT)

print(f"ROOT = {ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms ready (train: augmented | val: clean resize)")

In [ ]:
# ── Cell 2: Dataset & DataLoaders ─────────────────────────────────────────────
import json
import random as _random
import unicodedata
from pathlib import Path as _Path

from torch.utils.data import random_split, DataLoader, Dataset
from PIL import Image as _PILImage


def _nfc(s: str) -> str:
    """Normalize a string to NFC (fixes macOS NFD vs Linux NFC mismatch)."""
    return unicodedata.normalize("NFC", s)


def _find_image_path(images_dirs, image_id, category=None):
    """Search for image_id across multiple image root directories.
    All path components are NFC-normalized to handle cross-platform Unicode.
    """
    image_id_n = _nfc(image_id)
    category_n = _nfc(category) if category else None

    for images_dir in images_dirs:
        if not images_dir.exists():
            continue
        if category_n:
            for subdir_name in [f"Category:{category_n}", category_n]:
                path = images_dir / subdir_name / image_id_n
                if path.exists():
                    return path
        for subdir in images_dir.iterdir():
            if subdir.is_dir():
                subdir_n = images_dir / _nfc(subdir.name)
                path = subdir_n / image_id_n
                if path.exists():
                    return path
                path = subdir / image_id_n
                if path.exists():
                    return path
    return None


class HeritageDataset(Dataset):
    """Base dataset: returns (PIL_Image, caption_str, all_captions_list).
    No transform applied here — use TransformWrapper for train/val transforms.
    """

    def __init__(self, json_path, images_dir):
        json_path = _Path(json_path)
        if isinstance(images_dir, (str, _Path)):
            self.images_dirs = [_Path(images_dir)]
        else:
            self.images_dirs = [_Path(d) for d in images_dir]

        with open(json_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        self.valid_entries = []
        skipped = 0
        for entry in metadata:
            path = _find_image_path(
                self.images_dirs, entry["image_id"], entry.get("category")
            )
            if path is not None:
                self.valid_entries.append((entry, path))
            else:
                skipped += 1
        if skipped:
            print(f"  ({skipped} entries skipped — image not found on disk)")

    def __len__(self):
        return len(self.valid_entries)

    def __getitem__(self, idx):
        entry, image_path = self.valid_entries[idx]
        captions = entry.get("captions", [])
        caption  = _random.choice(captions) if captions else ""
        image    = _PILImage.open(image_path).convert("RGB")
        return image, caption


class TransformWrapper(Dataset):
    """Wraps a Subset (from random_split) and applies a transform to images."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, caption = self.subset[idx]
        return self.transform(image), caption


# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE   = 32     # T4 16 GB VRAM handles 32 with mixed precision; drop to 16 if OOM
MAX_CAP_LEN  = 50
VAL_FRACTION = 0.10
LR           = 3e-4
NUM_EPOCHS   = 25
PATIENCE     = 5      # early stopping patience
SEED         = 42

DATA_DIR  = ROOT / "data"
CKPT_DIR  = ROOT / "outputs" / "checkpoints"
FIG_DIR   = ROOT / "outputs" / "figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load dataset (no transform — PIL images) ─────────────────────────────────
print("Loading dataset ...")
full_dataset = HeritageDataset(
    json_path  = DATA_DIR / "processed" / "metadata_merged.json",
    images_dir = [
        DATA_DIR / "raw" / "wikimedia" / "images",
        DATA_DIR / "raw" / "danam" / "images",
    ],
)
print(f"Total valid images: {len(full_dataset)}")

n_val   = int(len(full_dataset) * VAL_FRACTION)
n_train = len(full_dataset) - n_val
train_subset, val_subset = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_dataset = TransformWrapper(train_subset, train_transform)
val_dataset   = TransformWrapper(val_subset,   val_transform)
print(f"Split  →  train: {n_train}  |  val: {n_val}")

# ── Collate: tokenize captions, mask padding with -100 ───────────────────────
def collate_fn(samples):
    images   = torch.stack([s[0] for s in samples])
    captions = [s[1] for s in samples]

    encoded = tokenizer(
        captions,
        padding="max_length",
        truncation=True,
        max_length=MAX_CAP_LEN,
        return_tensors="pt",
    )
    input_ids      = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100
    return images, input_ids, attention_mask, labels

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)
print(f"DataLoaders  →  train batches: {len(train_loader)}  |  val batches: {len(val_loader)}")
print(f"Batch size: {BATCH_SIZE}  |  Workers: 4")

In [ ]:
# ── Cell 3: Model Architecture + Optimizer + Scheduler ────────────────────────

# --- 1. ENCODER (ResNet50 — frozen) ---
class HeritageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights="ResNet50_Weights.DEFAULT")
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, images):
        features = self.resnet(images)                              # [B, 2048, 7, 7]
        features = features.permute(0, 2, 3, 1)                    # [B, 7, 7, 2048]
        return features.view(features.size(0), -1, features.size(-1))  # [B, 49, 2048]


# --- 2. ATTENTION (Bahdanau / Additive) ---
class HeritageAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.W = nn.Linear(decoder_dim, attention_dim)
        self.U = nn.Linear(encoder_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1)

    def forward(self, features, hidden):
        # features: [B, 49, 2048]   hidden: [B, 768]
        hidden_with_time = hidden.unsqueeze(1)                      # [B, 1, 768]
        score   = torch.tanh(self.W(hidden_with_time) + self.U(features))  # [B, 49, attn_dim]
        weights = torch.softmax(self.v(score), dim=1)               # [B, 49, 1]
        context = torch.sum(weights * features, dim=1)              # [B, 2048]
        return context, weights


# --- 3. COMPLETE MODEL ---
class HeritageLens(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder   = HeritageEncoder()
        self.attention = HeritageAttention(encoder_dim=2048, decoder_dim=768, attention_dim=512)
        self.gpt2      = GPT2LMHeadModel.from_pretrained("gpt2")

        self.query_projection   = nn.Linear(2048, 768)   # mean pool → attention query
        self.context_projection = nn.Linear(2048, 768)   # attention output → GPT-2 dim
        self.dropout = nn.Dropout(0.3)

    def forward(self, images, captions, labels=None):
        features = self.encoder(images)                             # [B, 49, 2048]

        # Attention: query from mean-pooled features, attend over spatial grid
        query = self.query_projection(features.mean(dim=1))         # [B, 768]
        context, _ = self.attention(features, query)                # [B, 2048]
        visual_context = self.dropout(self.context_projection(context))  # [B, 768]

        # Inject visual context into word embeddings
        inputs_embeds   = self.gpt2.transformer.wte(captions)       # [B, seq_len, 768]
        combined_embeds = inputs_embeds + visual_context.unsqueeze(1)

        loss_labels = labels if labels is not None else captions
        outputs = self.gpt2(inputs_embeds=combined_embeds, labels=loss_labels)
        return outputs.loss, outputs.logits


# ── Instantiate ──────────────────────────────────────────────────────────────
model = HeritageLens(vocab_size=tokenizer.vocab_size).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (encoder frozen)")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-2,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler("cuda")

print(f"\nOptimizer: AdamW  LR={LR}  Scheduler: CosineAnnealing  T_max={NUM_EPOCHS}")
print("Model ready.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ── Autoregressive generation helper (for real BLEU eval) ─────────────────────
@torch.no_grad()
def _generate_from_tensor(img_tensor, max_length=50):
    """Generate a caption from a pre-processed image tensor on device."""
    model.eval()
    img = img_tensor.unsqueeze(0).to(device) if img_tensor.dim() == 3 else img_tensor.to(device)
    ids = torch.tensor([[tokenizer.bos_token_id]], device=device)
    generated = []
    for _ in range(max_length):
        _, logits = model(img, ids)
        nxt = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(0)
        if nxt.item() == tokenizer.eos_token_id:
            break
        generated.append(nxt.item())
        ids = torch.cat([ids, nxt], dim=-1)
    return tokenizer.decode(generated, skip_special_tokens=True)

# ── Training configuration ────────────────────────────────────────────────────
BLEU_SAMPLE_N = 20   # generate real captions on this many val images per epoch
MAX_GRAD_NORM = 1.0
history       = {"train_loss": [], "val_loss": [], "val_bleu": [], "lr": []}
best_val_loss = float("inf")
patience_ctr  = 0
smoother      = SmoothingFunction().method1

print(f"Training: {NUM_EPOCHS} epochs | batch {BATCH_SIZE} | LR {LR} → cosine | patience {PATIENCE}")

for epoch in range(NUM_EPOCHS):
    current_lr = optimizer.param_groups[0]["lr"]
    history["lr"].append(current_lr)

    # ── 1. TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")

    for batch in pbar:
        images, input_ids, _, labels = [x.to(device) for x in batch]
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss, _ = model(images, input_ids, labels=labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        total_train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.2e}")

    avg_train_loss = total_train_loss / len(train_loader)
    history["train_loss"].append(avg_train_loss)

    # ── 2. VALIDATE (loss) ────────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Val loss"):
            images, input_ids, _, labels = [x.to(device) for x in batch]
            with torch.amp.autocast("cuda"):
                loss, _ = model(images, input_ids, labels=labels)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)
    history["val_loss"].append(avg_val_loss)

    # ── 3. REAL BLEU (autoregressive generation on N val images) ──────────────
    bleu_scores = []
    sample_indices = _random.sample(range(len(val_dataset)), min(BLEU_SAMPLE_N, len(val_dataset)))
    for si in sample_indices:
        img_t, ref_caption = val_dataset[si]
        gen_caption = _generate_from_tensor(img_t)
        ref_tokens  = ref_caption.split()
        gen_tokens  = gen_caption.split()
        if gen_tokens and ref_tokens:
            bleu_scores.append(
                sentence_bleu([ref_tokens], gen_tokens,
                              weights=(0.25, 0.25, 0.25, 0.25),
                              smoothing_function=smoother)
            )
    avg_bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    history["val_bleu"].append(avg_bleu)

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}  "
          f"LR: {current_lr:.2e}  "
          f"Train: {avg_train_loss:.4f}  "
          f"Val: {avg_val_loss:.4f}  "
          f"BLEU-4: {avg_bleu:.4f}")

    # ── 4. CHECKPOINT + EARLY STOPPING ────────────────────────────────────────
    torch.save({
        "epoch": epoch + 1, "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_loss": avg_val_loss, "val_bleu": avg_bleu,
    }, CKPT_DIR / f"epoch_{epoch+1:02d}.pt")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), CKPT_DIR / "best_model.pt")
        print(f"  -> Best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

    scheduler.step()

# ── 5. PLOT ───────────────────────────────────────────────────────────────────
n_actual = len(history["train_loss"])
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"],   label="Val Loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history["val_bleu"], color="green", label="BLEU-4 (generated)")
axes[1].set_title("BLEU-4 (autoregressive)"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(history["lr"], color="orange", label="Learning Rate")
axes[2].set_title("LR Schedule"); axes[2].set_xlabel("Epoch"); axes[2].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", dpi=150)
plt.show()

print(f"\nTraining complete after {n_actual} epochs.")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best model:    {CKPT_DIR / 'best_model.pt'}")

In [ ]:
# ── Cell 5: Inference & Evaluation ────────────────────────────────────────────
from PIL import Image as _InfImage
import random as _inf_random
import torch.nn.functional as F

# ── Generation functions ──────────────────────────────────────────────────────

@torch.no_grad()
def generate_caption(image_path, max_length=50, mode="greedy", top_p=0.9, temperature=0.7):
    """Generate a caption from an image file path.
    mode: 'greedy' or 'nucleus' (top-p sampling).
    """
    model.eval()
    img = _InfImage.open(image_path).convert("RGB")
    img_t = val_transform(img).unsqueeze(0).to(device)
    return _decode(img_t, max_length, mode, top_p, temperature)


@torch.no_grad()
def _decode(img_tensor, max_length, mode, top_p, temperature):
    ids = torch.tensor([[tokenizer.bos_token_id]], device=device)
    generated = []
    for _ in range(max_length):
        _, logits = model(img_tensor, ids)
        next_logits = logits[:, -1, :]

        if mode == "nucleus":
            next_logits = next_logits / temperature
            probs = F.softmax(next_logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            mask = cum_probs - sorted_probs > top_p
            sorted_probs[mask] = 0.0
            sorted_probs /= sorted_probs.sum()
            pick = torch.multinomial(sorted_probs, 1)
            nxt = sorted_idx.gather(-1, pick)
        else:
            nxt = torch.argmax(next_logits, dim=-1).unsqueeze(-1)

        if nxt.item() == tokenizer.eos_token_id:
            break
        generated.append(nxt.item())
        ids = torch.cat([ids, nxt], dim=-1)

    return tokenizer.decode(generated, skip_special_tokens=True)


# ── Load best checkpoint ─────────────────────────────────────────────────────
best_ckpt = CKPT_DIR / "best_model.pt"
if best_ckpt.exists():
    model.load_state_dict(torch.load(best_ckpt, map_location=device, weights_only=True))
    print(f"Loaded best model from: {best_ckpt}")
else:
    print("No checkpoint found — using current model weights.")

# ── Evaluate on 8 random val images: grid with ground truth comparison ───────
N_SHOW = 8
val_indices = val_subset.indices
sample_idx  = _inf_random.sample(list(val_indices), min(N_SHOW, len(val_indices)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
bleu_scores = []

for ax_i, ds_idx in enumerate(sample_idx):
    entry, img_path = full_dataset.valid_entries[ds_idx]
    gt_captions = entry.get("captions", [])
    gt_short    = gt_captions[0][:80] + "..." if gt_captions and len(gt_captions[0]) > 80 else (gt_captions[0] if gt_captions else "N/A")

    img = _InfImage.open(img_path).convert("RGB")
    img_t = val_transform(img).unsqueeze(0).to(device)

    gen_greedy  = _decode(img_t, 50, "greedy", 0.9, 0.7)
    gen_nucleus = _decode(img_t, 50, "nucleus", 0.9, 0.7)

    if gt_captions:
        ref_tokens = gt_captions[0].split()
        gen_tokens = gen_greedy.split()
        if ref_tokens and gen_tokens:
            bleu = sentence_bleu([ref_tokens], gen_tokens,
                                 weights=(0.25, 0.25, 0.25, 0.25),
                                 smoothing_function=smoother)
            bleu_scores.append(bleu)

    axes[ax_i].imshow(img)
    axes[ax_i].axis("off")
    axes[ax_i].set_title(
        f"GT: {gt_short}\n"
        f"Greedy: {gen_greedy[:80]}\n"
        f"Nucleus: {gen_nucleus[:80]}",
        fontsize=7, wrap=True, va="top"
    )

plt.suptitle("Inference: 8 Validation Images", fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "inference_grid.png", dpi=150)
plt.show()

avg_bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
print(f"\nAverage BLEU-4 on {len(bleu_scores)} shown images: {avg_bleu:.4f}")

# ── Full validation BLEU (autoregressive) ─────────────────────────────────────
print(f"\nComputing BLEU on full validation set ({len(val_indices)} images) ...")
full_bleu = []
for vi in tqdm(val_indices, desc="Generating captions"):
    entry, img_path = full_dataset.valid_entries[vi]
    gt_caps = entry.get("captions", [])
    if not gt_caps:
        continue
    img = _InfImage.open(img_path).convert("RGB")
    img_t = val_transform(img).unsqueeze(0).to(device)
    gen = _decode(img_t, 50, "greedy", 0.9, 0.7)
    refs = [c.split() for c in gt_caps]
    gen_t = gen.split()
    if gen_t and refs:
        full_bleu.append(
            sentence_bleu(refs, gen_t,
                          weights=(0.25, 0.25, 0.25, 0.25),
                          smoothing_function=smoother)
        )

final_bleu = float(np.mean(full_bleu)) if full_bleu else 0.0
print(f"Full validation BLEU-4: {final_bleu:.4f}  ({len(full_bleu)} images)")
